In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision.transforms as T
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

import numpy as np
import pandas as pd
from PIL import Image
import ast
import math
import random

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

C:\fyp-manish-shyam-phase-2\torch_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:

# Token / model dims
D_MODEL = 512          # Transformer hidden size
IMG_EMBED_DIM = 512
KG_EMBED_DIM = 512
TEXT_EMBED_DIM = 512

# Transformer
N_HEADS = 8
N_DECODER_LAYERS = 6
D_FF = 2048
DROPOUT = 0.1

# Training
BATCH_SIZE = 4
MAX_REPORT_LEN = 150
LR = 3e-4

NUM_WORKERS = 0   # Windows-safe

print("D_MODEL:", D_MODEL)


D_MODEL: 512


In [3]:

image_transform = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [4]:

LOCATION_TOKENS = {
    "Head": ("<HEAD_BOS>", "<HEAD_EOS>"),
    "Thorax": ("<THORAX_BOS>", "<THORAX_EOS>"),
    "Abdomen": ("<ABDOMEN_BOS>", "<ABDOMEN_EOS>"),
    "Spine and Muscles": ("<SPINE_BOS>", "<SPINE_EOS>"),
    "Reproductive and Urinary System": ("<GU_BOS>", "<GU_EOS>")
}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

special_tokens = {
    "pad_token": "<PAD>",
    "additional_special_tokens": []
}

for bos, eos in LOCATION_TOKENS.values():
    special_tokens["additional_special_tokens"].extend([bos, eos])

tokenizer.add_special_tokens(special_tokens)

VOCAB_SIZE = len(tokenizer)
PAD_ID = tokenizer.pad_token_id

print("Vocab size:", VOCAB_SIZE)


Vocab size: 30533


In [5]:

class MedPixDataset(Dataset):
    def __init__(self, csv_path, dataset_root, transform=None):
        self.df = pd.read_csv(csv_path).fillna("")
        self.transform = transform
        self.dataset_root = dataset_root

    def __len__(self):
        return len(self.df)

    def _parse_list(self, s):
        if s == "" or s == "[]":
            return []
        return ast.literal_eval(s)

    # def _load_image(self, path):
    #     img = Image.open(path).convert("RGB")
    #     if self.transform:
    #         img = self.transform(img)
    #     return img
    
    def _load_image(self, path):
        """
        Remap old absolute paths to current dataset root.
        """
        # Extract only the filename
        filename = os.path.basename(path)
        # Construct new correct path
        new_path = os.path.join(self.dataset_root, IMAGES_DIR, filename)
        if not os.path.exists(new_path):
            raise FileNotFoundError(f"Image not found: {new_path}")
        img = Image.open(new_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img


    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        ct_imgs = [self._load_image(p) for p in self._parse_list(row["CT_image_paths"])]
        mri_imgs = [self._load_image(p) for p in self._parse_list(row["MRI_image_paths"])]

        return {
            "uid": row["U_id"],
            "ct_images": ct_imgs,
            "mri_images": mri_imgs,
            "text_input": row["combined_clean"],
            "report": row["findings"],
            "location": row["Location Category"]
        }


In [6]:
import os

class ImageEncoder(nn.Module):
    """
    Converts a set of images into image tokens.
    Output: (B, N_images, D_MODEL)
    """
    def __init__(self, d_model=D_MODEL):
        super().__init__()

        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Remove classification head
        self.cnn = nn.Sequential(*list(base.children())[:-2])  # (B, 512, H/32, W/32)

        for p in self.cnn.parameters():
            p.requires_grad = False  # frozen initially

        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        """
        x: (B, N, 3, H, W)
        returns: (B, N, D_MODEL)
        """
        B, N, C, H, W = x.shape
        x = x.view(B * N, C, H, W)

        feat = self.cnn(x)                    # (B*N, 512, h, w)
        feat = feat.mean(dim=[2, 3])          # global avg pool → (B*N, 512)

        feat = self.proj(feat)                # (B*N, D_MODEL)
        feat = feat.view(B, N, -1)             # (B, N, D_MODEL)

        return feat


In [7]:

class TextEncoder(nn.Module):
    """
    Encodes auxiliary clinical text into token embeddings.
    Output: (B, T_text, D_MODEL)
    """
    def __init__(self, model_name="bert-base-uncased", d_model=D_MODEL):
        super().__init__()

        self.lm = AutoModel.from_pretrained(model_name)
        self.lm.resize_token_embeddings(len(tokenizer))  # 🔑 REQUIRED
        hidden = self.lm.config.hidden_size

        for p in self.lm.parameters():
            p.requires_grad = False  # frozen initially

        self.proj = nn.Linear(hidden, d_model)

    def forward(self, input_ids, attention_mask):
        """
        input_ids: (B, T)
        attention_mask: (B, T)
        """
        out = self.lm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        tokens = out.last_hidden_state          # (B, T, H)
        tokens = self.proj(tokens)              # (B, T, D_MODEL)

        return tokens


In [8]:

def pad_image_tokens(batch_imgs):
    """
    batch_imgs: list[list[Tensor]] length B
    returns:
      imgs: (B, N_max, 3, H, W)
      mask: (B, N_max)  (1 = valid, 0 = pad)
    """
    B = len(batch_imgs)
    max_n = max(len(x) for x in batch_imgs)

    padded_imgs = []
    masks = []

    for imgs in batch_imgs:
        if len(imgs) == 0:
            dummy = torch.zeros(3, 224, 224)
            imgs = [dummy]

        pad = max_n - len(imgs)
        padded = imgs + [torch.zeros_like(imgs[0])] * pad
        mask = [1] * len(imgs) + [0] * pad

        padded_imgs.append(torch.stack(padded))
        masks.append(torch.tensor(mask))

    return torch.stack(padded_imgs), torch.stack(masks)


In [9]:

def collate_fn(batch):
    # -------- IDs --------
    uids = [b["uid"] for b in batch]

    # -------- Images --------
    ct_imgs, ct_mask = pad_image_tokens([b["ct_images"] for b in batch])
    mri_imgs, mri_mask = pad_image_tokens([b["mri_images"] for b in batch])

    # -------- Text input --------
    text_inputs = [b["text_input"] for b in batch]
    text_enc = tokenizer(
        text_inputs,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # -------- Target reports --------
    reports = [b["report"] for b in batch]
    report_enc = tokenizer(
        reports,
        padding=True,
        truncation=True,
        max_length=MAX_REPORT_LEN,
        return_tensors="pt"
    )

    locations = [b["location"] for b in batch]

    return {
        "uid": uids,
        "ct_images": ct_imgs,
        "ct_mask": ct_mask,
        "mri_images": mri_imgs,
        "mri_mask": mri_mask,
        "text_input_ids": text_enc["input_ids"],
        "text_attention_mask": text_enc["attention_mask"],
        "report_ids": report_enc["input_ids"],
        "report_mask": report_enc["attention_mask"],
        "locations": locations
    }


In [10]:

ct_encoder = ImageEncoder().to(DEVICE)
mri_encoder = ImageEncoder().to(DEVICE)
text_encoder = TextEncoder().to(DEVICE)

print("Encoders initialized")

# ---- Quick shape sanity check ----
dummy_imgs = torch.randn(2, 3, 3, 224, 224).to(DEVICE)
dummy_tokens = ct_encoder(dummy_imgs)
print("Image tokens shape:", dummy_tokens.shape)


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Encoders initialized
Image tokens shape: torch.Size([2, 3, 512])


In [11]:

KG_LOCATION_MAP = {
    "Head": r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\split_by_location_category_matrices\Head_matrix.csv",
    "Thorax": r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\split_by_location_category_matrices\Thorax_matrix.csv",
    "Abdomen": r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\split_by_location_category_matrices\Abdomen_matrix.csv",
    "Spine and Muscles": r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\split_by_location_category_matrices\Spine_and_Muscles_matrix.csv",
    "Reproductive and Urinary System": r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\split_by_location_category_matrices\Reproductive_and_Urinary_System_matrix.csv"
}


def normalize_adjacency(A: torch.Tensor):
    """
    A: (N, N)
    returns: (N, N) normalized adjacency with self-loops
    """
    N = A.size(0)
    A = A + torch.eye(N, device=A.device)
    D = A.sum(dim=1)
    D_inv_sqrt = torch.pow(D, -0.5)
    D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0
    D_inv_sqrt = torch.diag(D_inv_sqrt)
    return D_inv_sqrt @ A @ D_inv_sqrt


A_hat_dict = {}

for loc, path in KG_LOCATION_MAP.items():
    df = pd.read_csv(path, index_col=0)
    A = torch.tensor(df.values, dtype=torch.float32, device=DEVICE)
    A_hat_dict[loc] = normalize_adjacency(A)
    print(f"{loc}: KG nodes = {A.shape[0]}")


Head: KG nodes = 4400
Thorax: KG nodes = 4400
Abdomen: KG nodes = 4400
Spine and Muscles: KG nodes = 4400
Reproductive and Urinary System: KG nodes = 4400


In [12]:

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, A_hat, X):
        return F.relu(self.linear(A_hat @ X))


class GCN(nn.Module):
    """
    Produces node embeddings:
    Output: (N_nodes, D_MODEL)
    """
    def __init__(self, in_dim, hidden_dim, out_dim, num_layers=2):
        super().__init__()

        dims = [in_dim] + [hidden_dim] * (num_layers - 1) + [out_dim]
        self.layers = nn.ModuleList([
            GCNLayer(dims[i], dims[i + 1])
            for i in range(len(dims) - 1)
        ])

    def forward(self, A_hat, X):
        for layer in self.layers:
            X = layer(A_hat, X)
        return X   # (N_nodes, D_MODEL)


In [13]:

# Infer node count from any location
example_loc = next(iter(A_hat_dict))
N_NODES = A_hat_dict[example_loc].shape[0]

# Identity node features (one-hot)
X_nodes = torch.eye(N_NODES, device=DEVICE)

print("KG node feature shape:", X_nodes.shape)


KG node feature shape: torch.Size([4400, 4400])


In [14]:

class KGEncoder(nn.Module):
    """
    Converts KG into node tokens for a given location.
    Output: (N_nodes, D_MODEL)
    """
    def __init__(self, gcn):
        super().__init__()
        self.gcn = gcn

    def forward(self, location):
        A_hat = A_hat_dict[location]
        kg_tokens = self.gcn(A_hat, X_nodes)
        return kg_tokens


In [15]:

gcn = GCN(
    in_dim=N_NODES,
    hidden_dim=256,
    out_dim=D_MODEL,
    num_layers=2
).to(DEVICE)

kg_encoder = KGEncoder(gcn).to(DEVICE)

# ---- Sanity check ----
kg_tokens = kg_encoder("Head")
print("KG tokens shape:", kg_tokens.shape)


KG tokens shape: torch.Size([4400, 512])


In [16]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: (B, T, D_MODEL)
        """
        return x + self.pe[:, :x.size(1)]


In [17]:

# class ReportEmbedding(nn.Module):
#     def __init__(self, vocab_size, d_model, dropout=0.1):
#         super().__init__()
#         self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
#         self.pos = PositionalEncoding(d_model)
#         self.dropout = nn.Dropout(dropout)

#     def forward(self, input_ids):
#         """
#         input_ids: (B, T)
#         """
#         x = self.embed(input_ids) * math.sqrt(D_MODEL)
#         x = self.pos(x)
#         return self.dropout(x)

# =========================
# Cell 17 (FIXED): Decoder Embedding (BERT-tied)
# =========================

class ReportEmbedding(nn.Module):
    def __init__(self, tokenizer, text_encoder, d_model, dropout=0.1):
        super().__init__()

        # 🔑 Tie embeddings to BERT
        self.embed = text_encoder.lm.get_input_embeddings()

        self.proj = nn.Linear(
            self.embed.embedding_dim,
            d_model,
            bias=False
        )

        self.pos = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        x = self.embed(input_ids)
        x = self.proj(x)
        x = self.pos(x)
        return self.dropout(x)


In [18]:

def build_memory(ct_tokens, mri_tokens, text_tokens, kg_tokens):
    """
    Concatenate all non-autoregressive tokens into ONE memory bank.

    ct_tokens:  (B, N_ct, D)
    mri_tokens: (B, N_mri, D)
    text_tokens:(B, T_text, D)
    kg_tokens:  (B, N_nodes, D)

    returns:
      memory: (B, M, D)
    """

    memory = torch.cat(
        [ct_tokens, mri_tokens, text_tokens, kg_tokens],
        dim=1
    )

    return memory


In [19]:

class ReportDecoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = ReportEmbedding(tokenizer, text_encoder, D_MODEL, DROPOUT)


        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            dropout=DROPOUT,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=N_DECODER_LAYERS
        )

        self.output_proj = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, input_ids, memory, tgt_key_padding_mask=None):
        """
        input_ids: (B, T)
        memory:    (B, M, D)
        """

        tgt = self.embedding(input_ids)

        # causal mask (autoregressive)
        T = input_ids.size(1)
        causal_mask = torch.triu(
            torch.ones(T, T, device=input_ids.device),
            diagonal=1
        ).bool()

        out = self.decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )

        return self.output_proj(out)  # (B, T, vocab)


In [20]:

decoder = ReportDecoder().to(DEVICE)
print("Transformer decoder initialized")

# ---- Dummy sanity check ----
B = 2
T = 10

dummy_ids = torch.randint(0, VOCAB_SIZE, (B, T)).to(DEVICE)
dummy_mem = torch.randn(B, 20, D_MODEL).to(DEVICE)

out = decoder(dummy_ids, dummy_mem)
print("Decoder output shape:", out.shape)


Transformer decoder initialized
Decoder output shape: torch.Size([2, 10, 30533])


In [21]:

def get_bos_ids(locations):
    """
    locations: list[str] length B
    returns: (B,) tensor of BOS token IDs
    """
    return torch.tensor(
        [
            tokenizer.convert_tokens_to_ids(LOCATION_TOKENS[loc][0])
            for loc in locations
        ],
        device=DEVICE
    )


In [22]:
def get_kg_tokens_batch(locations):
    unique_locs = list(set(locations))
    loc_to_tokens = {
        loc: kg_encoder(loc) for loc in unique_locs
    }
    return torch.stack([loc_to_tokens[loc] for loc in locations])


In [23]:

def forward_step(batch):
    """
    batch: output of collate_fn
    returns: logits, targets
    """

    # -------- Images --------
    ct_imgs = batch["ct_images"].to(DEVICE)
    mri_imgs = batch["mri_images"].to(DEVICE)

    ct_mask = batch["ct_mask"].to(DEVICE)
    mri_mask = batch["mri_mask"].to(DEVICE)

    ct_tokens = ct_encoder(ct_imgs)
    mri_tokens = mri_encoder(mri_imgs)

    # -------- Text --------
    text_ids = batch["text_input_ids"].to(DEVICE)
    text_mask = batch["text_attention_mask"].to(DEVICE)

    text_tokens = text_encoder(text_ids, text_mask)

    # -------- KG --------
    kg_tokens = get_kg_tokens_batch(batch["locations"])


    # -------- Memory --------
    memory = build_memory(ct_tokens, mri_tokens, text_tokens, kg_tokens)

    # -------- Reports --------
    report_ids = batch["report_ids"].to(DEVICE)
    report_mask = batch["report_mask"].to(DEVICE)

    B = report_ids.size(0)

    bos_ids = get_bos_ids(batch["locations"]).unsqueeze(1)

    # Decoder input: [BOS] + report[:-1]
    decoder_input = torch.cat(
        [bos_ids, report_ids[:, :-1]],
        dim=1
    )

    targets = report_ids  # predict actual tokens

    logits = decoder(
        decoder_input,
        memory,
        tgt_key_padding_mask=(decoder_input == PAD_ID)
    )

    return logits, targets


In [24]:
# Loss Function

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=0.1
)


In [25]:
#Optimizer
# Freeze encoders initially
for m in [ct_encoder, mri_encoder, text_encoder, gcn]:
    for p in m.parameters():
        p.requires_grad = False

# Train decoder first
params = list(decoder.parameters())

optimizer = torch.optim.AdamW(
    params,
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

print("Optimizer ready")


Optimizer ready


In [26]:
# Training loop function
from tqdm import tqdm

def train_one_epoch(loader):
    decoder.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc="Training", leave=True)

    for batch in pbar:
        optimizer.zero_grad()

        logits, targets = forward_step(batch)

        loss = criterion(
            logits.reshape(-1, VOCAB_SIZE),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()
    return total_loss / len(loader)


In [27]:
# Instantiate dataset and create dataloaders======

csv_path = r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\data\df_overall.csv"
DATASET_ROOT = r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\dataset\MedPix-2.0\MedPix-2-0"
IMAGES_DIR = "images"

full_dataset = MedPixDataset(
    csv_path=csv_path,
    dataset_root = DATASET_ROOT,
    transform=image_transform
)

print("Total samples:", len(full_dataset))


Total samples: 671


In [28]:

from torch.utils.data import random_split

train_ratio = 0.9
train_size = int(train_ratio * len(full_dataset))
test_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, test_size],
    generator=generator
)

print("Train:", len(train_dataset))
print("Test :", len(test_dataset))


Train: 603
Test : 68


In [29]:

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True
)

print("DataLoaders ready")


DataLoaders ready


In [30]:
# Sanity Check Batch

batch = next(iter(train_loader))

print("UIDs:", batch["uid"][:3])
print("CT images:", batch["ct_images"].shape)
print("MRI images:", batch["mri_images"].shape)
print("Text IDs:", batch["text_input_ids"].shape)
print("Report IDs:", batch["report_ids"].shape)
print("Locations:", batch["locations"][:3])


UIDs: ['MPX1923', 'MPX1709', 'MPX2492']
CT images: torch.Size([4, 1, 3, 224, 224])
MRI images: torch.Size([4, 2, 3, 224, 224])
Text IDs: torch.Size([4, 119])
Report IDs: torch.Size([4, 119])
Locations: ['Spine and Muscles', 'Spine and Muscles', 'Abdomen']


In [31]:
# sanity check
batch = next(iter(train_loader))
logits, targets = forward_step(batch)

print("Logits:", logits.shape)
print("Targets:", targets.shape)


Logits: torch.Size([4, 150, 30533])
Targets: torch.Size([4, 150])


In [32]:
# sample check to see if loss is correctly backpropagated

loss = criterion(
    logits.reshape(-1, VOCAB_SIZE),
    targets.reshape(-1)
)

loss.backward()

print("Loss:", loss.item())


Loss: 10.492368698120117


In [33]:
# # Training pipeline for epochs begins here

# NUM_EPOCHS = 5

# for epoch in range(NUM_EPOCHS):
#     train_loss = train_one_epoch(train_loader)
#     print(
#         f"[Stage 1] Epoch {epoch+1}/{NUM_EPOCHS} | "
#         f"Train Loss: {train_loss:.4f}"
#     )


In [34]:
# # Freeze decoder cross-attention (syntax warmup)
# for layer in decoder.decoder.layers:
#     layer.multihead_attn.requires_grad_(False)


In [35]:
# for epoch in range(2):
#     loss = train_one_epoch(train_loader)
#     print(f"[Warmup] Epoch {epoch+1} | Loss {loss:.4f}")


In [36]:
# for layer in decoder.decoder.layers:
#     layer.multihead_attn.requires_grad_(True)


In [37]:
# DECODER ONLY TRAINING FIRST

NUM_EPOCHS = 8

# Freeze ALL encoders
for m in [ct_encoder, mri_encoder, text_encoder, gcn]:
    for p in m.parameters():
        p.requires_grad = False

# Unfreeze decoder fully
for p in decoder.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(
    decoder.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=4,
    gamma=0.5
)

print("Stage 1: Decoder language warmup")


Stage 1: Decoder language warmup


In [38]:
for epoch in range(NUM_EPOCHS): # 8 epochs (DECODER ONLY)
    loss = train_one_epoch(train_loader)
    print(f"[Stage 1] Epoch {epoch+1}/{NUM_EPOCHS} | Loss {loss:.4f}")


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [01:13<00:00,  2.04it/s, loss=7.6401]


[Stage 1] Epoch 1/8 | Loss 7.5601


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.21it/s, loss=7.0353]


[Stage 1] Epoch 2/8 | Loss 7.1684


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.19it/s, loss=7.0812]


...

[Stage 1] Epoch 6/8 | Loss 6.4921


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.16it/s, loss=6.4671]


[Stage 1] Epoch 7/8 | Loss 6.3239


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.15it/s, loss=6.2789]

[Stage 1] Epoch 8/8 | Loss 6.2130


In [39]:
# Freeze decoder self-attention
for layer in decoder.decoder.layers:
    layer.self_attn.requires_grad_(False)

# Keep cross-attention trainable
for layer in decoder.decoder.layers:
    layer.multihead_attn.requires_grad_(True)

print("Stage 2: Cross-attention warmup")


Stage 2: Cross-attention warmup


In [40]:
for epoch in range(3):
    loss = train_one_epoch(train_loader)
    print(f"[Stage 2] Epoch {epoch+1}/3 | Loss {loss:.4f}")


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:46<00:00,  3.27it/s, loss=6.0305]


[Stage 2] Epoch 1/3 | Loss 6.0633


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.17it/s, loss=5.9545]


[Stage 2] Epoch 2/3 | Loss 5.9795


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:48<00:00,  3.09it/s, loss=6.0796]

[Stage 2] Epoch 3/3 | Loss 5.9290


In [41]:
# Unfreeze entire decoder
for layer in decoder.decoder.layers:
    layer.self_attn.requires_grad_(True)
    layer.multihead_attn.requires_grad_(True)

print("Stage 3: Full decoder training")


Stage 3: Full decoder training


In [42]:
for epoch in range(12):
    loss = train_one_epoch(train_loader)
    print(f"[Stage 3] Epoch {epoch+1}/12 | Loss {loss:.4f}")


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.16it/s, loss=5.9180]


[Stage 3] Epoch 1/12 | Loss 5.8693


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.16it/s, loss=5.5936]


[Stage 3] Epoch 2/12 | Loss 5.7869


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:48<00:00,  3.14it/s, loss=6.0983]


...

[Stage 3] Epoch 10/12 | Loss 5.5615


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:48<00:00,  3.14it/s, loss=5.4148]


[Stage 3] Epoch 11/12 | Loss 5.5554


Training: 100%|█████████████████████████████████████████████████████████| 151/151 [00:47<00:00,  3.17it/s, loss=5.6766]

[Stage 3] Epoch 12/12 | Loss 5.5329


In [43]:
#decoder pipeline begins here

def calc_banned_tokens_ngram(prev_tokens, no_repeat_ngram_size):
    """
    prev_tokens: list[int] generated so far
    returns: set[int] token ids to ban at next step
    """
    if len(prev_tokens) < no_repeat_ngram_size - 1:
        return set()

    ngrams = {}
    for i in range(len(prev_tokens) - no_repeat_ngram_size + 1):
        prefix = tuple(prev_tokens[i:i + no_repeat_ngram_size - 1])
        next_tok = prev_tokens[i + no_repeat_ngram_size - 1]
        ngrams.setdefault(prefix, set()).add(next_tok)

    current_prefix = tuple(prev_tokens[-(no_repeat_ngram_size - 1):])
    return ngrams.get(current_prefix, set())


In [44]:
# Beam Search Decoder

@torch.no_grad()
def beam_search_decode(
    memory,
    location,
    beam_size=5,
    max_len=MAX_REPORT_LEN,
    min_len=25,
    repetition_penalty=1.4,
    no_repeat_ngram_size=4
):
    """
    memory: (1, M, D_MODEL)
    location: str
    returns: decoded string
    """

    decoder.eval()

    bos_id = tokenizer.convert_tokens_to_ids(LOCATION_TOKENS[location][0])
    eos_id = tokenizer.convert_tokens_to_ids(LOCATION_TOKENS[location][1])

    # Each beam = (token_ids, log_prob)
    beams = [([bos_id], 0.0)]

    for step in range(max_len):
        new_beams = []

        for tokens, score in beams:

            # If EOS already generated, keep beam
            if tokens[-1] == eos_id:
                new_beams.append((tokens, score))
                continue

            input_ids = torch.tensor(
                tokens, device=DEVICE
            ).unsqueeze(0)  # (1, T)

            logits = decoder(input_ids, memory)[0, -1]  # (vocab,)

            # -----------------------------
            # Repetition penalty
            # -----------------------------
            for t in set(tokens):
                logits[t] /= repetition_penalty

            # -----------------------------
            # No-repeat ngram constraint
            # -----------------------------
            if no_repeat_ngram_size > 0 and len(tokens) >= no_repeat_ngram_size:
                banned = set()
                prefix = tokens[-(no_repeat_ngram_size - 1):]

                for i in range(len(tokens) - no_repeat_ngram_size + 1):
                    if tokens[i:i + no_repeat_ngram_size - 1] == prefix:
                        banned.add(tokens[i + no_repeat_ngram_size - 1])

                logits[list(banned)] = -1e9

            # -----------------------------
            # Prevent early EOS
            # -----------------------------
            if step < min_len:
                logits[eos_id] = -1e9

            log_probs = torch.log_softmax(logits, dim=-1)
            topk = torch.topk(log_probs, beam_size)

            for i in range(beam_size):
                new_tokens = tokens + [topk.indices[i].item()]
                new_score = score + topk.values[i].item()
                new_beams.append((new_tokens, new_score))

        # Keep top beams
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

        # Stop early if all beams ended
        if all(b[0][-1] == eos_id for b in beams):
            break

    best_tokens = beams[0][0]

    return tokenizer.decode(
        best_tokens,
        skip_special_tokens=True
    )


In [45]:

@torch.no_grad()
def generate_one(batch, idx):
    """
    batch: a collated batch
    idx: index within batch
    """

    ct_tokens = ct_encoder(batch["ct_images"].to(DEVICE))
    mri_tokens = mri_encoder(batch["mri_images"].to(DEVICE))

    text_tokens = text_encoder(
        batch["text_input_ids"].to(DEVICE),
        batch["text_attention_mask"].to(DEVICE)
    )

    kg_tokens = get_kg_tokens_batch(batch["locations"])

    memory = build_memory(
        ct_tokens[idx:idx+1],
        mri_tokens[idx:idx+1],
        text_tokens[idx:idx+1],
        kg_tokens[idx:idx+1]
    )

    return beam_search_decode(
        memory=memory,
        location=batch["locations"][idx],
        beam_size=5,               
        repetition_penalty=1.4,     
        no_repeat_ngram_size=4,     
        min_len=25            
    )


In [49]:
from tqdm import tqdm
import pandas as pd

def run_inference(loader):
    decoder.eval()
    results = []

    for batch in tqdm(loader, desc="Running inference"):
        B = len(batch["locations"])
        for i in range(B):
            gen = generate_one(batch, i)
            gt = tokenizer.decode(
                batch["report_ids"][i],
                skip_special_tokens=True
            )
            results.append({
                "uid": batch["uid"][i],
                "location": batch["locations"][i],
                "generated_report": gen,
                "ground_truth_report": gt
            })

    return pd.DataFrame(results)


In [47]:
# simple check to see if its working

batch = next(iter(test_loader))

gen = generate_one(batch, idx=0)

gt = tokenizer.decode(
    batch["report_ids"][0],
    skip_special_tokens=True
)

print("LOCATION:", batch["locations"][0])
print("\nGENERATED:\n", gen)
print("\nGROUND TRUTH:\n", gt[:500])


LOCATION: Thorax

GENERATED:
 c - opal sub thickening, andular pattern with. basillyalalalal scatteredalalal mualalal pre fi.ening and honey orbropha noac,sti " bi intersis ofaratalcos abnormal re patternle thick subly pre

GROUND TRUTH:
 hrct chest : peripheral / basilar honeycombing, irregular intralobular septal thickening, irregular interlobular septal thickening, patchy ground glass


In [50]:
# Run inference on test set
results_df = run_inference(test_loader)

print("Total samples generated:", len(results_df))
results_df.head()


Running inference: 100%|███████████████████████████████████████████████████████████████| 17/17 [08:57<00:00, 31.60s/it]

Total samples generated: 68


,uid,location,generated_report,ground_truth_report
0,MPX1473,Thorax,"c - opal sub thickening, andular pattern with....",hrct chest : peripheral / basilar honeycombing...
1,MPX2000,Head,• intra mass the mass of spinal mass mass post...,well defined centrally located mass within the...
2,MPX2084,Head,"multiple brain the lesions temporal in, ofric ...",multiple lesions within the brain. the most pr...
3,MPX1094,Head,"• there, the is intra the the the and.,,,, the...",there is an encephalocele in the frontal regio...
4,MPX2544,Thorax,mri : of is theric in with. of of of of the of...,abnormal left ventricular dilatation with nonc...


In [51]:
results_df.to_csv(r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\results\revamped\raw_results_after_basic_cleaning.csv")

In [52]:
import torch
import os
import json
from datetime import datetime

CKPT_PATH = r"D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\results\checkpoints\model_stage1_partial.pt"
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

checkpoint = {
    # =========================
    # Model states
    # =========================
    "decoder_state": decoder.state_dict(),
    "ct_encoder_state": ct_encoder.state_dict(),
    "mri_encoder_state": mri_encoder.state_dict(),
    "text_encoder_state": text_encoder.state_dict(),
    "gcn_state": gcn.state_dict(),

    # =========================
    # Optimizer / scheduler
    # =========================
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),

    # =========================
    # Tokenizer info
    # =========================
    "tokenizer_name": "bert-base-uncased",
    "special_tokens": tokenizer.special_tokens_map,
    "added_tokens": tokenizer.get_added_vocab(),

    # =========================
    # Training metadata
    # =========================
    "meta": {
        "stage": "partial multimodal training",
        "notes": "Before heavy cleaning + LM pretraining",
        "date": datetime.now().isoformat(),
        "D_MODEL": D_MODEL,
        "N_DECODER_LAYERS": N_DECODER_LAYERS,
        "VOCAB_SIZE": VOCAB_SIZE
    }
}

torch.save(checkpoint, CKPT_PATH)

print(f"✅ Checkpoint saved to {CKPT_PATH}")


✅ Checkpoint saved to D:\fyp-manish-shyam-pahse-2-dataset\Medpix-2.0-Medical-Report-Generation-Using-Deep-Learning\results\checkpoints\model_stage1_partial.pt
